In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


In [56]:
train = pd.read_csv("../data/csv/train.csv")
test  = pd.read_csv("../data/csv/test.csv")

target = "is_fraud"

In [57]:
train.head()

,timestamp,transaction_id,account_id,hour_of_day,day_of_week,is_weekend,amount,merchant_category,mcc_code,merchant_country,...,ip_risk_score,is_foreign_txn,time_since_last_s,velocity_1h,amount_vs_avg_ratio,account_age_days,has_2fa,credit_limit,is_fraud,fraud_pattern
0,2022-01-01 00:00:01,TXN000277317,ACC0045807,0,5,1,65.78,utilities,4900,US,...,21.4,0,232,1,0.5266,1773,1,3849.06,0,NaN
1,2022-01-01 00:02:54,TXN000667824,ACC0028794,0,5,1,64.01,grocery,5411,DE,...,40.6,1,15,1,3.4563,2408,0,1742.87,0,NaN
2,2022-01-01 00:03:29,TXN000766996,ACC0005585,0,5,1,801.70,travel,4511,DE,...,24.2,1,36,0,11.4923,1692,1,5923.23,0,NaN
3,2022-01-01 00:05:39,TXN000525597,ACC0000492,0,5,1,352.71,hotel,7011,CA,...,34.0,1,462,0,3.8205,3171,0,3544.65,0,NaN
4,2022-01-01 00:06:10,TXN000089982,ACC0033651,0,5,1,43.13,grocery,5411,US,...,45.4,0,184,1,0.7770,1255,0,2517.79,0,NaN


In [146]:
test.head()

,timestamp,transaction_id,account_id,hour_of_day,day_of_week,is_weekend,amount,merchant_category,mcc_code,merchant_country,...,ip_risk_score,is_foreign_txn,time_since_last_s,velocity_1h,amount_vs_avg_ratio,account_age_days,has_2fa,credit_limit,is_fraud,fraud_pattern
0,2024-01-01 00:01:02,TXN000821117,ACC0007684,0,0,0,70.16,gas_station,5541,US,...,21.4,0,219,0,0.7257,2356,1,36625.77,0,NaN
1,2024-01-01 00:01:28,TXN000518955,ACC0045728,0,0,0,47.81,pharmacy,5912,RO,...,35.2,1,42,3,0.5861,136,1,6993.76,0,NaN
2,2024-01-01 00:03:14,TXN000370107,ACC0029408,0,0,0,96.37,online_retail,5999,US,...,14.2,0,103,1,1.1530,90,1,2993.71,0,NaN
3,2024-01-01 00:03:15,TXN000498540,ACC0038965,0,0,0,73.81,pharmacy,5912,US,...,12.4,0,467,1,1.8686,2133,1,5091.72,0,NaN
4,2024-01-01 00:05:41,TXN000804374,ACC0043488,0,0,0,18.41,grocery,5411,US,...,11.0,0,249,0,0.1352,2901,1,4781.95,0,NaN


In [58]:
train = train.drop(columns=['fraud_pattern', 'transaction_id', 'account_id'], errors='ignore')
test = test.drop(columns=['fraud_pattern', 'transaction_id', 'account_id'], errors='ignore')

In [59]:
train['timestamp'] = pd.to_datetime(train['timestamp'])
train.sort_values(by='timestamp', inplace = True)
train.set_index('timestamp', inplace=True)

In [60]:
test['timestamp'] = pd.to_datetime(test['timestamp'])
test.sort_values(by='timestamp', inplace = True)
test.set_index('timestamp', inplace=True)

In [61]:
#Safe Division and Safe Logarithm functions
def safe_divide(num, denom, fill=0):
    return (num / (denom.replace(0, 1e-10))).replace([np.inf,-np.inf], fill).fillna(fill)

def safe_log(s, fill=0):
    return pd.Series(np.log1p(np.maximum(s, 0)), index=s.index).replace([np.inf,-np.inf], fill).fillna(fill)

In [62]:
# Fit quantile-based parameters for risk amount features
def fit_params(df):
    params = {}

    # Amount
    params['amount_q95'] = df['amount'].quantile(0.95)

    # Risk
    if "ip_risk_score" in df.columns:
        params['ip_q80'] = df['ip_risk_score'].quantile(0.80)

    if "amount_vs_avg_ratio" in df.columns:
        params['ratio_q90'] = df['amount_vs_avg_ratio'].quantile(0.90)

    if "velocity_1h" in df.columns:
        params['velocity_q90'] = df['velocity_1h'].quantile(0.90)

    if "credit_limit" in df.columns:
        util = df['amount'] / df['credit_limit'].replace(0, np.nan)
        params['util_q90'] = util.quantile(0.90)

    print("✅ Quantile params fitted")
    return params

In [63]:
#Add features
def add_features(df, params):
    df = df.copy()
    new_var = []
    
    # 1. AMOUNT FEATURES
    if 'amount' in df.columns:
        df['log_amount'] = safe_log(df['amount'])
        df['is_round_amount'] = ((df['amount'] % 10 == 0) & (df['amount'] > 0)).astype(int)
        df['high_amount_flag'] = (df['amount'] > params['amount_q95']).astype(int)

        new_var += ['log_amount', 'is_round_amount', 'high_amount_flag']

    # 2. TIME FEATURES
    if 'hour_of_day' in df.columns:
        
        df['is_night'] = df['hour_of_day'].between(0, 5).astype(int)
        df['is_business_hours'] = df['hour_of_day'].between(9, 17).astype(int)

        new_var += ['is_night', 'is_business_hours']

    # 3. RISK FEATURES
    # 1) IP risk score features
    if "ip_risk_score" in df.columns and 'ip_q80' in params:
        df["log_ip_risk_score"] = safe_log(df["ip_risk_score"])
        df["high_ip_risk_flag"] = (df["ip_risk_score"] > params['ip_q80']).astype(int)
        new_var += ["log_ip_risk_score", "high_ip_risk_flag"]
        df.drop(columns=["ip_risk_score"], inplace=True)

    if "amount_vs_avg_ratio" in df.columns and 'ratio_q90' in params:
        df["log_amount_vs_avg_ratio"] = safe_log(df["amount_vs_avg_ratio"])
        df["high_amount_vs_avg_flag"] = (df["amount_vs_avg_ratio"] > params['ratio_q90']).astype(int)
        new_var += ["log_amount_vs_avg_ratio", "high_amount_vs_avg_flag"]
        df.drop(columns=["amount_vs_avg_ratio"], inplace=True)

    if "velocity_1h" in df.columns and 'velocity_q90' in params:
        df["log_velocity_1h"] = safe_log(df["velocity_1h"])
        df["high_velocity_1h_flag"] = (df["velocity_1h"] > params['velocity_q90']).astype(int)
        new_var += ["log_velocity_1h", "high_velocity_1h_flag"]
        df.drop(columns=["velocity_1h"], inplace=True)

    if "credit_limit" in df.columns and 'util_q90' in params:
        df["utilization"] = safe_divide(df["amount"], df["credit_limit"])
        df["high_utilization_flag"] = (df["utilization"] > params['util_q90']).astype(int)
        new_var += ["utilization", "high_utilization_flag"]
        df.drop(columns=['amount'], inplace=True)
        df.drop(columns=["credit_limit"], inplace=True)

    print(f"✅ Features added: {len(new_var)}")

    return df, new_var

In [64]:
# fit
params = fit_params(train)

✅ Quantile params fitted


**Logistic Regression Encode**

In [65]:
num_cols = ['account_age_days', 'time_since_last_s', 'log_amount', 'is_round_amount', 'high_amount_flag', 'log_ip_risk_score', 'high_ip_risk_flag', 'log_amount_vs_avg_ratio', 'high_amount_vs_avg_flag', 'log_velocity_1h', 'high_velocity_1h_flag', 'utilization', 'high_utilization_flag']
cate_cols = ['merchant_category', 'merchant_country', 'device_type', 'mcc_code', 'hour_of_day', 'day_of_week']
binary_cols = ['is_fraud', 'is_weekend', 'card_present', 'device_known', 'is_foreign_txn', 'has_2fa']

In [66]:
#Onehot Encode for categorical variables
def fit_onehot(df, cate_cols):
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    encoded = encoder.fit_transform(df[cate_cols])

    feature_names = encoder.get_feature_names_out(cate_cols)
    encoded_df = pd.DataFrame(encoded, columns=feature_names, index=df.index)

    df = df.drop(columns=cate_cols)
    df = pd.concat([df, encoded_df], axis=1)

    return df, encoder

def transform_onehot(df, cate_cols, encoder):
    encoded = encoder.transform(df[cate_cols])

    feature_names = encoder.get_feature_names_out(cate_cols)
    encoded_df = pd.DataFrame(encoded, columns=feature_names, index=df.index)

    df = df.drop(columns=cate_cols)
    df = pd.concat([df, encoded_df], axis=1)

    return df

In [67]:
#Scale numeric variables
def scale_numeric(df, target='is_fraud'):
    df = df.copy()
    
    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])
    
    print(f"✅ Scaled {len(num_cols)} numeric columns")
    
    return df, scaler, num_cols

In [68]:
train_log = train.copy()

# 1. Feature engineering
train_log, new_var = add_features(train_log, params)

# 2. One-hot FIT
train_log, encoder = fit_onehot(train_log, cate_cols)

# lấy lại categorical columns sau encoding
cat_cols = cate_cols

# 3. Scaling FIT
train_log, scaler, num_cols = scale_numeric(train_log, target)

# 4. Save schema
train_columns = train_log.columns

train_log.to_csv("train_log_processed.csv", index=False)

test_log = test.copy()

# 1. Feature engineering (same params)
test_log, _ = add_features(test_log, params)

# 2. One-hot TRANSFORM (IMPORTANT FIX)
test_log = transform_onehot(test_log, cate_cols, encoder)

# 3. Align columns
test_log = test_log.reindex(columns=train_columns, fill_value=0)

# 4. Scaling TRANSFORM ONLY
test_log[num_cols] = scaler.transform(test_log[num_cols])

test_log.to_csv("test_log_processed.csv", index=False)

print("✅ Test processing done")

✅ Features added: 13
✅ Scaled 13 numeric columns
✅ Features added: 13
✅ Test processing done


In [69]:
train_log.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 666085 entries, 2022-01-01 00:00:01 to 2023-12-31 23:59:42
Data columns (total 96 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   is_weekend                        666085 non-null  int64  
 1   card_present                      666085 non-null  int64  
 2   device_known                      666085 non-null  int64  
 3   is_foreign_txn                    666085 non-null  int64  
 4   time_since_last_s                 666085 non-null  float64
 5   account_age_days                  666085 non-null  float64
 6   has_2fa                           666085 non-null  int64  
 7   is_fraud                          666085 non-null  int64  
 8   log_amount                        666085 non-null  float64
 9   is_round_amount                   666085 non-null  float64
 10  high_amount_flag                  666085 non-null  float64
 11  is_night                     

In [70]:
test_log.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 166413 entries, 2024-01-01 00:01:02 to 2024-06-30 23:59:53
Data columns (total 96 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   is_weekend                        166413 non-null  int64  
 1   card_present                      166413 non-null  int64  
 2   device_known                      166413 non-null  int64  
 3   is_foreign_txn                    166413 non-null  int64  
 4   time_since_last_s                 166413 non-null  float64
 5   account_age_days                  166413 non-null  float64
 6   has_2fa                           166413 non-null  int64  
 7   is_fraud                          166413 non-null  int64  
 8   log_amount                        166413 non-null  float64
 9   is_round_amount                   166413 non-null  float64
 10  high_amount_flag                  166413 non-null  float64
 11  is_night                     

In [139]:
#train_log.to_parquet(".../train_log_processed.parquet", index=False)

In [140]:
#test_log.to_parquet(".../test_log_processed.parquet", index=False)

**Tree Models Encode**

In [71]:
def label_encode(df, encoders=None, target='is_fraud'):
    df = df.copy()

    if encoders is None:
        encoders = {}
        
        for col in cate_cols:
            
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
            print(f"✅ Fit & encoded '{col}' ({df[col].nunique()} cats)")
    
    else:
        for col in cate_cols:
            if col in encoders:
                le = encoders[col]

                # handle unseen labels
                df[col] = df[col].astype(str)
                df[col] = df[col].map(lambda x: x if x in le.classes_ else 'UNKNOWN')

                # add UNKNOWN to encoder if needed
                if 'UNKNOWN' not in le.classes_:
                    le.classes_ = np.append(le.classes_, 'UNKNOWN')

                df[col] = le.transform(df[col])
            else:
                df[col] = 0  # fallback

    return df, encoders

In [72]:
# TRAIN
train_tree = train.copy()
train_tree, new_var = add_features(train, params)

train_tree, encoders = label_encode(train_tree, target=target)

train_columns = train_tree.columns

train_tree.to_csv("train_tree_processed.csv", index=False)

# TEST
test_tree = test.copy()
test_tree, _ = add_features(test_tree, params)

test_tree, _ = label_encode(test_tree, encoders=encoders, target=target)

# align columns
test_tree = test_tree.reindex(columns=train_columns, fill_value=0)

test_tree.to_csv("test_tree_processed.csv", index=False)

✅ Features added: 13
✅ Fit & encoded 'merchant_category' (14 cats)
✅ Fit & encoded 'merchant_country' (11 cats)
✅ Fit & encoded 'device_type' (5 cats)
✅ Fit & encoded 'mcc_code' (14 cats)
✅ Fit & encoded 'hour_of_day' (24 cats)
✅ Fit & encoded 'day_of_week' (7 cats)
✅ Features added: 13


In [73]:
train_tree.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 666085 entries, 2022-01-01 00:00:01 to 2023-12-31 23:59:42
Data columns (total 27 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   hour_of_day              666085 non-null  int64  
 1   day_of_week              666085 non-null  int64  
 2   is_weekend               666085 non-null  int64  
 3   merchant_category        666085 non-null  int64  
 4   mcc_code                 666085 non-null  int64  
 5   merchant_country         666085 non-null  int64  
 6   card_present             666085 non-null  int64  
 7   device_type              666085 non-null  int64  
 8   device_known             666085 non-null  int64  
 9   is_foreign_txn           666085 non-null  int64  
 10  time_since_last_s        666085 non-null  int64  
 11  account_age_days         666085 non-null  int64  
 12  has_2fa                  666085 non-null  int64  
 13  is_fraud                 666085 non-

In [74]:
test_tree.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 166413 entries, 2024-01-01 00:01:02 to 2024-06-30 23:59:53
Data columns (total 27 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   hour_of_day              166413 non-null  int64  
 1   day_of_week              166413 non-null  int64  
 2   is_weekend               166413 non-null  int64  
 3   merchant_category        166413 non-null  int64  
 4   mcc_code                 166413 non-null  int64  
 5   merchant_country         166413 non-null  int64  
 6   card_present             166413 non-null  int64  
 7   device_type              166413 non-null  int64  
 8   device_known             166413 non-null  int64  
 9   is_foreign_txn           166413 non-null  int64  
 10  time_since_last_s        166413 non-null  int64  
 11  account_age_days         166413 non-null  int64  
 12  has_2fa                  166413 non-null  int64  
 13  is_fraud                 166413 non-

In [141]:
#train_tree.to_parquet(".../train_tree_processed.parquet", index=False)

In [142]:
#test_tree.to_parquet(".../test_tree_processed.parquet", index=False)